# Institutional Portfolio & Risk Analytics Dashboard

# Notebook 1: 
Download Historical Stock Data

# Objective: 
Download historical stock price data using Python and prepare it for portfolio analysis.

# Author:
Ekta Lanjewar

# workflow

Import libraries
        ↓
Choose portfolio
        ↓
Download data
        ↓
Check currency and history
        ↓
Inspect dataset
        ↓
Check missing values
        ↓
Extract Adj Close
        ↓
Handle missing values
        ↓
Verify data quality
        ↓
Download and align the benchmark
        ↓
Save data

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Nine eurozone companies, all listed and traded in EUR
tickers = [
    "ASML.AS",   # ASML Holding - Euronext Amsterdam
    "BNP.PA",    # BNP Paribas - Euronext Paris
    "MC.PA",     # LVMH - Euronext Paris
    "SIE.DE",    # Siemens - Xetra (Frankfurt)
    "AIR.PA",    # Airbus - Euronext Paris
    "OR.PA",     # L'Oréal - Euronext Paris
    "SAN.PA",    # Sanofi - Euronext Paris
    "SAP.DE",    # SAP - Xetra (Frankfurt)
    "ALV.DE"     # Allianz - Xetra (Frankfurt)
]

# Benchmark: Xtrackers EURO STOXX 50 UCITS ETF 1C (Xetra, EUR).
# It is an accumulating fund (dividends are reinvested) that tracks the
# EURO STOXX 50 Net Return index. All nine holdings belong to this index.
benchmark = "XESC.DE"

start_date = "2019-01-01"
end_date = "2025-12-31"
data = yf.download(tickers, start=start_date, end=end_date, auto_adjust=False)

data.head()

## Check that every ticker is traded in EUR

The portfolio is analysed from the perspective of an **unhedged euro-based investor holding euro-denominated shares**. Because every holding and the benchmark are priced in EUR, no currency conversion is needed and returns contain no exchange-rate effect.

We confirm the trading currency that Yahoo Finance reports for each ticker. The notebook stops if any ticker is not in EUR.

In [ ]:
for ticker in tickers + [benchmark]:
    currency = yf.Ticker(ticker).fast_info["currency"]
    print(f"{ticker:8} {currency}")
    assert currency == "EUR", f"{ticker} is not traded in EUR"

# Check object type
type(data)

# Check dataset dimensions
data.shape

# View column names
data.columns

# View row index (dates)
data.index

# Summarize dataset
data.info()

# Count missing values
data.isna().sum()

In [ ]:
data.shape

1. The number of rows is the number of trading days in the sample (one row per date on which at least one of the exchanges was open).
2. For each stock, Yahoo Finance returned multiple fields:
Open
High
Low
Close
Adj Close
Volume

Since you downloaded 9 stocks, there are:

6 fields × 9 stocks = 54 columns

In [ ]:
data.columns

In [ ]:
type(data)

In [ ]:
data.index #This tells us the index contains dates and times. "Every row in this DataFrame is identified by a date.

## Data Inspection (data.info)

In this section, we inspect the downloaded dataset to understand:

- its structure,
- data types,
- missing values (NaN),
- and overall quality before performing any analysis.

In [ ]:
data.info()

## Missing Values

The portfolio contains stocks listed on Euronext Paris, Euronext Amsterdam and Xetra (Frankfurt). These exchanges share most holidays, but not all of them (for example, Xetra is closed on 24 and 31 December, when Euronext is open), so a few dates contain missing observations. We inspect these missing values before deciding on a cleaning strategy.

# Check for missing (NaN) values in each column.
# isna() identifies missing values, and sum() counts them.
# This helps assess data quality before analysis.

In [ ]:
data.isna().sum()

## Select Adjusted Closing Prices

Portfolio returns should be calculated using Adjusted Closing Prices because they account for stock splits and dividend adjustments.

In [ ]:
adj_close= data['Adj Close']
adj_close

## Check the historical coverage and continuity of each stock

Each stock must have prices for the whole period from January 2019 to December 2025, with no long gaps.

The dates in the table are the union of the trading days of Euronext Paris, Euronext Amsterdam and Xetra. A day on which all three exchanges were closed (for example Christmas Day) is not in the table at all. A stock therefore only has a missing price when **its own exchange was closed while another was open**, for example Xetra on 24 and 31 December. These normal exchange holidays create gaps of one or two trading days.

For each stock we report the first and last available price, the number of missing values and the **longest run of consecutive missing prices**. A run longer than five trading days cannot be explained by an exchange holiday and points to a data problem (or a trading suspension), so the notebook stops.

In [ ]:
def longest_missing_run(prices):
    """Longest number of consecutive dates with a missing price."""
    longest = 0
    current = 0
    for is_missing in prices.isna():
        if is_missing:
            current += 1
            longest = max(longest, current)
        else:
            current = 0
    return longest


coverage = pd.DataFrame({
    "first_price": adj_close.apply(lambda s: s.first_valid_index().date()),
    "last_price": adj_close.apply(lambda s: s.last_valid_index().date()),
    "missing_values": adj_close.isna().sum(),
    "longest_missing_run": adj_close.apply(longest_missing_run)
})
print(coverage)

# Every stock must start at the beginning and end at the end of the sample
assert (adj_close.apply(lambda s: s.first_valid_index()) == adj_close.index[0]).all(), "A stock starts late"
assert (adj_close.apply(lambda s: s.last_valid_index()) == adj_close.index[-1]).all(), "A stock ends early"

# Normal exchange holidays create gaps of one or two trading days
max_gap = 5
assert (coverage["longest_missing_run"] <= max_gap).all(), "A stock has more than 5 consecutive missing prices"

## Handle Missing Values

The portfolio contains stocks listed on different exchanges, which observe different trading holidays. As a result, some dates contain missing (`NaN`) prices.

Using `dropna()` would remove every row containing at least one missing value, causing unnecessary loss of valid market data.

Instead, `ffill()` (forward fill) replaces each missing value with the most recent available price, preserving the time series for portfolio analysis.

### Example

Before:

| Date | Siemens |
|------|---------|
| 23 Dec | 230 |
| 24 Dec | NaN |
| 25 Dec | NaN |
| 26 Dec | 235 |

After `ffill()`:

| Date | Siemens |
|------|---------|
| 23 Dec | 230 |
| 24 Dec | 230 |
| 25 Dec | 230 |
| 26 Dec | 235 |

In [ ]:
adj_close= adj_close.ffill()
adj_close.head() #Printing all 1,809 rows clutters the notebook. Usually we inspect only the beginning or end of the dataset.

In [ ]:
adj_close.isna().sum() 

## Download and align the benchmark

The benchmark is downloaded for the same period and aligned with the portfolio's dates:

- `reindex(adj_close.index)` keeps exactly the portfolio's dates, so the portfolio and the benchmark cover the same period.
- `ffill()` fills a date with no benchmark price (for example a day when Xetra was closed but Euronext was open) with the most recent previous price, the same treatment as the stock prices.
- Values are never filled backwards (`bfill`), because that would use future information (look-ahead bias).

We use the **Adjusted Close**, as for the stocks. Because the fund is accumulating, it does not pay dividends, so its Adjusted Close and Close should be almost identical.

In [ ]:
benchmark_data = yf.download(benchmark, start=start_date, end=end_date, auto_adjust=False)

benchmark_prices = benchmark_data["Adj Close"]
benchmark_close = benchmark_data["Close"]
# yfinance may return a one-column table instead of a Series
if isinstance(benchmark_prices, pd.DataFrame):
    benchmark_prices = benchmark_prices.iloc[:, 0]
    benchmark_close = benchmark_close.iloc[:, 0]
benchmark_prices = benchmark_prices.dropna()
benchmark_close = benchmark_close.dropna()

print("Benchmark data from", benchmark_prices.index.min().date(), "to", benchmark_prices.index.max().date())

# The benchmark must cover the whole portfolio period
assert benchmark_prices.index.min() <= adj_close.index.min(), "Benchmark starts after the portfolio"
assert benchmark_prices.index.max() >= adj_close.index.max(), "Benchmark ends before the portfolio"

## Validate the benchmark data

Three checks compare the downloaded benchmark with what we expect from an accumulating ETF. Unlike the earlier checks, a difference here does **not** stop the notebook: the expected values come from outside sources, so a difference is flagged for investigation and explained before the results are used.

**1. Adjusted Close versus Close.** An accumulating fund does not pay dividends, so Yahoo Finance should make no dividend adjustment and the two prices should be almost identical. A large difference would mean Yahoo recorded a distribution that the fund did not pay, or another data error.

**2. Calendar-year returns versus published fund returns.** The yearly returns calculated from the Yahoo prices are compared with the fund returns published by [justETF](https://www.justetf.com/en/etf-profile.html?isin=LU0380865021) (2019–2021 are not published there). Small differences are normal: published returns are usually based on the fund's official net asset value, while Yahoo uses Xetra closing prices, which can differ slightly from the net asset value. A difference above 0.5 percentage points is flagged.

In [ ]:
# 1. Adjusted Close versus Close
close_difference = (benchmark_prices / benchmark_close - 1).abs()
print(f"Largest difference between Adjusted Close and Close: {close_difference.max():.4%}")
if close_difference.max() > 0.001:
    print("INVESTIGATE: Adjusted Close differs from Close by more than 0.1% for an accumulating fund.")
else:
    print("OK: Adjusted Close and Close are almost identical, as expected for an accumulating fund.")

In [ ]:
# 2. Calendar-year returns versus published fund returns (justETF, in %)
published_returns = pd.Series({2022: -9.03, 2023: 22.80, 2024: 11.54, 2025: 21.77})

year_end_prices = benchmark_prices.groupby(benchmark_prices.index.year).last()
yahoo_returns = (year_end_prices.pct_change() * 100).round(2)

return_check = pd.DataFrame({
    "yahoo_return_%": yahoo_returns.loc[published_returns.index],
    "published_return_%": published_returns
})
return_check["difference_pp"] = return_check["yahoo_return_%"] - return_check["published_return_%"]
return_check["status"] = np.where(return_check["difference_pp"].abs() <= 0.5, "OK", "INVESTIGATE")
print(return_check)

In [ ]:
# Keep exactly the portfolio's dates, then fill gaps with the previous price only
benchmark_aligned = benchmark_prices.reindex(adj_close.index)
print("Portfolio dates with no benchmark price (filled with the previous day):", benchmark_aligned.isna().sum())

benchmark_aligned = benchmark_aligned.ffill()
benchmark_aligned.name = benchmark

assert benchmark_aligned.notna().all(), "Benchmark price missing on the first portfolio date"
assert benchmark_aligned.index.equals(adj_close.index)

## Check the largest daily moves: market movements or data errors?

Adjusted prices should reflect dividends, stock splits and spin-offs. A data error usually shows up as an unusually large one-day move. The table lists the three largest daily moves of each stock and of the benchmark, together with:

- the **benchmark's** and the **equal-weight portfolio's** return on the same day, and
- the stock's return on the **next** day.

How to read it:

- **Genuine market-wide movement:** the stock, the benchmark and the portfolio all move strongly in the same direction (for example March 2020).
- **Genuine stock-specific event:** the benchmark moves much less than the stock, and the move matches company news such as results or a spin-off. Check the date against the company's news.
- **Possible data error:** a large move that is reversed the next day, a large move with no company news, or a move on a spin-off or dividend date that does not match the value distributed.

Known events in this period include:

- **March 2020**: COVID-19 market crash (large moves in every stock).
- **28 September 2020 (SIE.DE)**: spin-off of Siemens Energy, one Siemens Energy share for every two Siemens shares. In the previously downloaded data, Siemens showed a gain of about +9% on this day. This is consistent with reports that the Siemens share price fell only slightly while shareholders also received the Siemens Energy shares, so the adjusted price reflects the spin-off.
- **6 May 2022 (SAN.PA)**: spin-off of EUROAPI, one EUROAPI share for every 23 Sanofi shares (less than 1% of Sanofi's value).
- **13 November 2025 (SIE.DE)**: Siemens shares fell almost 10% after its annual results.

Any other large move should be investigated before the data is used.

In [ ]:
# Daily returns of the nine stocks and of the benchmark, on the portfolio dates
all_returns = adj_close.copy()
all_returns[benchmark] = benchmark_aligned
all_returns = all_returns.pct_change().dropna()

portfolio_daily = all_returns[tickers].mean(axis=1)   # equal-weight portfolio

rows = []
for ticker in all_returns.columns:
    largest_moves = all_returns[ticker].abs().nlargest(3)
    for date in largest_moves.index:
        position = all_returns.index.get_loc(date)
        next_day = all_returns[ticker].iloc[position + 1] if position + 1 < len(all_returns) else np.nan
        rows.append({
            "ticker": ticker,
            "date": date.date(),
            "return_%": all_returns.loc[date, ticker] * 100,
            "benchmark_%": all_returns.loc[date, benchmark] * 100,
            "portfolio_%": portfolio_daily.loc[date] * 100,
            "next_day_%": next_day * 100
        })

largest_moves_table = pd.DataFrame(rows).round(1)
print(largest_moves_table.to_string(index=False))

# how to save as csv
1. ..      = Go back one folder
2. /data   = Enter the data folder
3. adj_close.csv = Save the file with this name

In [ ]:
import os

# Create the data folder if it does not exist yet (for example in a fresh copy of the project)
os.makedirs("../data", exist_ok=True)

# Stock prices in EUR (used by Notebooks 2 and 3 and the dashboard)
adj_close.to_csv("../data/adj_close.csv")

# Benchmark prices in EUR, on the same dates as the portfolio
benchmark_aligned.to_csv("../data/benchmark.csv")

In [ ]:
import os

os.listdir("../data")